# 02 — Multi-model e2e (Qwen + TinyLlama)

Target: `multi_model_serving` at `:8001` — `POST /generate`.

No per-token stream here — measure *end-to-end* latency, cold vs warm, and LRU churn across `qwen2.5-0.5b` and `tinyllama-1.1b`.

Start the multi-model lab first (blog: Serving Multiple Models).

## Prediction

> Cold first hit loads weights (seconds). Warm repeats should be much faster. Alternating models with LRU capacity 2 keeps both warm; forcing eviction (if capacity were 1) would reintroduce cold spikes.

Edit before running.

In [ ]:
import asyncio
import sys
from pathlib import Path

here = Path.cwd().resolve()
candidates = [
    here,
    here / "observability",
    here.parent / "observability",
    here.parent,
    here / "notebooks" / "llm-inference" / "observability",
]
for p in [here, *here.parents]:
    candidates.append(p / "notebooks" / "llm-inference" / "observability")
    candidates.append(p / "observability")

obs_root = None
seen: set[Path] = set()
for candidate in candidates:
    candidate = candidate.resolve()
    if candidate in seen:
        continue
    seen.add(candidate)
    if (candidate / "runner.py").is_file():
        sys.path.insert(0, str(candidate))
        sys.path.insert(0, str(candidate.parent))
        obs_root = candidate
        break

if obs_root is None:
    raise ModuleNotFoundError(f"observability/runner.py not found; cwd={here}")

from config import load_urls
from multi_client import clear_cache, generate, list_models
from runner import run_multi_e2e_batch, save_csv, save_json

urls = load_urls()
BASE = urls["MULTI_MODEL_URL"]
print("obs_root =", obs_root)
print("BASE =", BASE)
list_models(BASE)

In [ ]:
PROMPT = "What is continuous batching in one sentence?"
MODELS = ["qwen2.5-0.5b", "tinyllama-1.1b"]

# Cold: clear cache, then first generate per model
clear_cache(BASE)
cold_rows = []
for mid in MODELS:
    m = generate(BASE, mid, PROMPT, max_new_tokens=32)
    cold_rows.append({"phase": "cold", **m.to_dict()})
    print(mid, "cold e2e_s=", round(m.e2e_s, 3), "ok=", m.ok, "err=", m.error)

# Warm: same prompts again (both should be resident with LRU=2)
warm_rows = []
for mid in MODELS:
    m = generate(BASE, mid, PROMPT, max_new_tokens=32)
    warm_rows.append({"phase": "warm", **m.to_dict()})
    print(mid, "warm e2e_s=", round(m.e2e_s, 3), "ok=", m.ok)

In [ ]:
# Concurrent mixed-model e2e (both warm)
jobs = [
    ("qwen2.5-0.5b", "What is attention?"),
    ("tinyllama-1.1b", "What is a transformer?"),
    ("qwen2.5-0.5b", "What is speculative decoding?"),
    ("tinyllama-1.1b", "What is KV cache?"),
]

batch = await run_multi_e2e_batch(BASE, jobs, max_new_tokens=32)
print({k: batch[k] for k in batch if k != "requests"})
batch["requests"]

In [ ]:
import pandas as pd

df = pd.DataFrame(cold_rows + warm_rows)
display(df[["phase", "model_id", "e2e_s", "ok"]])

save_csv(cold_rows + warm_rows, "multi_cold_warm.csv")
save_json(batch, "multi_concurrent.json")
print("saved results/")

## Explain

1. Cold vs warm delta per model — is load time the dominant cost?
2. Under concurrent mixed traffic, did one model starve the other (shared GPU)?
3. How would TTFT look if multi-model grew a streaming path like single-model?

### Notes

- …